# День 4 — Mamba-2 (selective SSM)
Те же данные, лосс (= метрика) и валидация, что у GRU, — сравнение честное.

Перед запуском: **Среда выполнения → Сменить среду выполнения → T4 GPU**. Загрузите обновлённую папку `day2` на Drive поверх старой (новые файлы: `mamba.py`, `solution_template_mamba.py`, `bench_speed.py`).

**Планка:** у нашей GRU валидация 0.659 → лидерборд 0.644. Чтобы обогнать `baseline_x03` (0.6505 на лидерборде), Mamba нужно ~0.666+ на валидации.

## 1. GPU + Drive + код + быстрые тесты Mamba

In [ ]:
!nvidia-smi --query-gpu=name --format=csv,noheader
from google.colab import drive
drive.mount('/content/drive')
W = '/content/drive/MyDrive/wunder'
!rm -rf /content/day2 && cp -r $W/day2 /content/day2
%cd /content/day2
!python -m pytest -q tests/test_day2.py -k 'ssd or chunked or loss'

## 2. Данные (~12 мин; диск Colab очищается при каждом подключении)

In [ ]:
!python prepare_data.py --src $W/colab_data/valid_sub.parquet --out /content/data/valid
!python prepare_data.py --src $W/colab_data/train_sub.parquet --out /content/data/train

## 3. Пробный запуск (~5 мин): работает ли и сколько занимает эпоха
Смотрите на `M rows/s` и `eta`: эпоха на 3500 последовательностях = 70M строк.

In [ ]:
!python train.py --train-dir /content/data/train --valid-dir /content/data/valid --out /content/smoke_mamba \
    --arch mamba --epochs 1 --max-train-seqs 256 --valid-seqs 100 --lr 2e-3 --wd 0.01

## 4. Проверка скорости инференса против GRU, которая уже прошла лимит платформы
Обе модели меряются на одном CPU. Нужно **≤ 1.00x**. Если больше — обучайте с `--n-layers 1` (≈0.6x).

In [ ]:
!python export.py --ckpt /content/smoke_mamba/best.pt --out /content/smoke_mamba_sub --check-steps 2000
!python bench_speed.py $W/submissions/gru128x2_v1 /content/smoke_mamba_sub

## 5. Основное обучение (время = эпохи × время эпохи из ячейки 3)
Чекпоинты пишутся на Drive после каждой эпохи. Если сессия оборвалась: ячейки 1–2, затем эта же ячейка с `--resume` в конце команды.

In [ ]:
RUN = W + '/runs/mamba_v1'
!python train.py --train-dir /content/data/train --valid-dir /content/data/valid --out $RUN \
    --arch mamba --d-model 64 --n-layers 2 --d-state 16 --headdim 16 \
    --epochs 15 --batch 64 --chunk 1000 --lr 2e-3 --wd 0.01

## 6. Экспорт, проверка скорости и скачивание

In [ ]:
!python export.py --ckpt $RUN/best.pt --valid-dir /content/data/valid --out $W/submissions/mamba_v1
!python bench_speed.py $W/submissions/gru128x2_v1 $W/submissions/mamba_v1
from google.colab import files
files.download(W + '/submissions/mamba_v1.zip')